In [0]:
# =============================================================================
# SINGOLO CODICE: MAPPATURA + ESTRAZIONE DATI SOLO AFRICA (2000-2024)
# =============================================================================

from pyspark.sql import functions as F
import time
import random

# 1. Mappatura SOLO AFRICA (5 paesi × 3 città = 15 stazioni)
africa_stations = {
    "Africa": {
        "Sudafrica": [
            {"code": "688160", "city": "Cape Town (mediterraneo costiero)"},
            {"code": "683680", "city": "Johannesburg (altopiano subtropicale)"},
            {"code": "689940", "city": "Durban (subtropicale umido)"}
        ],
        "Nigeria": [
            {"code": "650460", "city": "Lagos (tropicale umido costiero)"},
            {"code": "650820", "city": "Kano (arido saheliano)"},
            {"code": "651670", "city": "Abuja (tropicale savana)"}
        ],
        "Egitto": [
            {"code": "623970", "city": "Il Cairo (deserto caldo)"},
            {"code": "624140", "city": "Alessandria (mediterraneo)"},
            {"code": "624200", "city": "Assuan (deserto estremo caldo)"}
        ],
        "Kenya": [
            {"code": "637400", "city": "Nairobi (altopiano tropicale)"},
            {"code": "637910", "city": "Mombasa (tropicale costiero)"},
            {"code": "636860", "city": "Eldoret (altopiano fresco)"}
        ],
        "Marocco": [
            {"code": "601550", "city": "Casablanca (mediterraneo atlantico)"},
            {"code": "602650", "city": "Marrakech (arido caldo interno)"},
            {"code": "602200", "city": "Tangeri (mediterraneo nord)"}
        ]
    }
}

# Flatten + mappa code → info città
usaf_to_city = {}
for paese, lista in africa_stations["Africa"].items():
    for item in lista:
        code = item["code"]
        usaf_to_city[code] = {
            "paese": paese,
            "city": item["city"]
        }

usaf_list = list(usaf_to_city.keys())
print(f"Stazioni Africa selezionate: {len(usaf_list)}")
print("Codici:", usaf_list)

# 2. Genera percorsi S3 (2000-2024)
years = list(range(2000, 2025))
paths_with_info = []
for year in years:
    for code in usaf_list:
        path = f"s3a://noaa-gsod-pds/{year}/{code}99999.csv"
        paths_with_info.append((path, code, year))

print(f"Totale percorsi generati: {len(paths_with_info)}")

# 3. Lettura sicura con try/except + anti-throttling
dfs = []
missing_files = []

for path, code, year in paths_with_info:
    time.sleep(random.uniform(0.4, 1.5))  # evita throttling S3
    try:
        df_temp = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(path)
        
        if df_temp.count() > 0:
            df_temp = df_temp.withColumn("USAF", F.lit(code)) \
                             .withColumn("year", F.lit(year))
            dfs.append(df_temp)
            print(f"✓ Letto: {path}")
        else:
            city = usaf_to_city.get(code, {}).get("city", "Sconosciuta")
            missing_files.append(f"{year} - {code} ({city}) → file vuoto")
    except Exception as e:
        city = usaf_to_city.get(code, {}).get("city", "Sconosciuta")
        missing_files.append(f"{year} - {code} ({city}) → {str(e)[:80]}...")

# 4. Unione dati validi
if dfs:
    df_africa = dfs[0]
    for d in dfs[1:]:
        df_africa = df_africa.unionByName(d, allowMissingColumns=True)
    
    # Aggiungi metadati
    df_africa = df_africa.withColumn("continent", F.lit("Africa")) \
                         .withColumn("season",
                                     F.when(F.month("DATE").isin([12,1,2]), "Inverno")
                                      .when(F.month("DATE").isin([3,4,5]), "Primavera")
                                      .when(F.month("DATE").isin([6,7,8]), "Estate")
                                      .otherwise("Autunno"))
    
    # Controllo temperature (converti se in °F)
    temp_avg = df_africa.agg(F.avg("TEMP").alias("avg")).collect()[0]["avg"]
    if temp_avg is not None and temp_avg > 40:
        print("→ Temperature in °F → conversione in °C")
        df_africa = df_africa.withColumn("TEMP_c",   (F.col("TEMP") - 32) * 5/9) \
                             .withColumn("MAX_c",    (F.col("MAX") - 32) * 5/9) \
                             .withColumn("MIN_c",    (F.col("MIN") - 32) * 5/9)
    else:
        df_africa = df_africa.withColumnRenamed("TEMP", "TEMP_c") \
                             .withColumnRenamed("MAX", "MAX_c") \
                             .withColumnRenamed("MIN", "MIN_c")

    # Filtra righe valide
    df_africa_clean = df_africa.filter(F.col("TEMP_c").isNotNull())
    
    print(f"\nAfrica completata — righe totali: {df_africa_clean.count()}")
    
    # 5. Salva in Delta (adatta al tuo Volume!)
    delta_path = "/Volumes/workspace/weather/continents/af/"  # ← CAMBIA CON IL TUO VOLUM
    df_africa_clean.write.format("delta").mode("overwrite").save(delta_path)
    print(f"Salvato in Delta: {delta_path}")
    
else:
    print("Nessun file valido per Africa!")

# 6. Report mancanti
if missing_files:
    print("\n=== FILE MANCANTI ===")
    print(f"Totale: {len(missing_files)}")
    for m in missing_files:
        print(m)
else:
    print("Tutti i file presenti o processati correttamente!")  

In [0]:
# =============================================================================
# SINGOLO CODICE: MAPPATURA + ESTRAZIONE DATI SOLO ASIA (2000-2024)
# =============================================================================

from pyspark.sql import functions as F
import time
import random

# 1. Mappatura SOLO ASIA (5 paesi × 3 città = 15 stazioni)
asia_stations = {
    "Asia": {
        "Cina": [
            {"code": "545110", "city": "Pechino (freddo continentale)"},
            {"code": "582380", "city": "Shanghai (umido subtropicale)"},
            {"code": "592870", "city": "Guangzhou (tropicale caldo)"}
        ],
        "India": [
            {"code": "421820", "city": "Delhi (arido caldo)"},
            {"code": "430630", "city": "Mumbai (monsonico costiero)"},
            {"code": "432790", "city": "Chennai (tropicale umido)"}
        ],
        "Giappone": [
            {"code": "476620", "city": "Tokyo (temperato umido)"},
            {"code": "477700", "city": "Osaka (simile a Tokyo)"},
            {"code": "479710", "city": "Naha Okinawa (subtropicale)"}
        ],
        "Russia": [
            {"code": "260380", "city": "Mosca (freddo continentale)"},
            {"code": "284500", "city": "Novosibirsk (siberiano estremo)"},
            {"code": "319600", "city": "Sochi (mar Nero, subtropicale)"}
        ],
        "Indonesia": [
            {"code": "967490", "city": "Jakarta (tropicale umido)"},
            {"code": "967410", "city": "Surabaya (tropicale)"},
            {"code": "971800", "city": "Denpasar Bali (tropicale)"}
        ]
    }
}

# Flatten + mappa code → info città
usaf_to_city = {}
for paese, lista in asia_stations["Asia"].items():
    for item in lista:
        code = item["code"]
        usaf_to_city[code] = {
            "paese": paese,
            "city": item["city"]
        }

usaf_list = list(usaf_to_city.keys())
print(f"Stazioni Asia selezionate: {len(usaf_list)}")
print("Codici:", usaf_list)

# 2. Genera percorsi S3 (2000-2024)
years = list(range(2000, 2025))
paths_with_info = []
for year in years:
    for code in usaf_list:
        path = f"s3a://noaa-gsod-pds/{year}/{code}99999.csv"
        paths_with_info.append((path, code, year))

print(f"Totale percorsi generati: {len(paths_with_info)}")

# 3. Lettura sicura con try/except + anti-throttling
dfs = []
missing_files = []

for path, code, year in paths_with_info:
    time.sleep(random.uniform(0.4, 1.5))  # evita throttling S3
    try:
        df_temp = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(path)
        
        if df_temp.count() > 0:
            df_temp = df_temp.withColumn("USAF", F.lit(code)) \
                             .withColumn("year", F.lit(year))
            dfs.append(df_temp)
            print(f"✓ Letto: {path}")
        else:
            city = usaf_to_city.get(code, {}).get("city", "Sconosciuta")
            missing_files.append(f"{year} - {code} ({city}) → file vuoto")
    except Exception as e:
        city = usaf_to_city.get(code, {}).get("city", "Sconosciuta")
        missing_files.append(f"{year} - {code} ({city}) → {str(e)[:80]}...")

# 4. Unione dati validi
if dfs:
    df_asia = dfs[0]
    for d in dfs[1:]:
        df_asia = df_asia.unionByName(d, allowMissingColumns=True)
    
    # Aggiungi metadati
    df_asia = df_asia.withColumn("continent", F.lit("Asia")) \
                     .withColumn("season",
                                 F.when(F.month("DATE").isin([12,1,2]), "Inverno")
                                  .when(F.month("DATE").isin([3,4,5]), "Primavera")
                                  .when(F.month("DATE").isin([6,7,8]), "Estate")
                                  .otherwise("Autunno"))
    
    # Controllo temperature (converti se in °F)
    temp_avg = df_asia.agg(F.avg("TEMP").alias("avg")).collect()[0]["avg"]
    if temp_avg is not None and temp_avg > 40:
        print("→ Temperature in °F → conversione in °C")
        df_asia = df_asia.withColumn("TEMP_c",   (F.col("TEMP") - 32) * 5/9) \
                         .withColumn("MAX_c",    (F.col("MAX") - 32) * 5/9) \
                         .withColumn("MIN_c",    (F.col("MIN") - 32) * 5/9)
    else:
        df_asia = df_asia.withColumnRenamed("TEMP", "TEMP_c") \
                         .withColumnRenamed("MAX", "MAX_c") \
                         .withColumnRenamed("MIN", "MIN_c")

    # Filtra righe valide
    df_asia_clean = df_asia.filter(F.col("TEMP_c").isNotNull())
    
    print(f"\nAsia completata — righe totali: {df_asia_clean.count()}")
    
    # 5. Salva in Delta (adatta al tuo Volume!)
    delta_path = "/Volumes/workspace/weather/continents/as/"  # ← CAMBIA CON IL TUO VOLUM
    df_asia_clean.write.format("delta").mode("overwrite").save(delta_path)
    print(f"Salvato in Delta: {delta_path}")
    
else:
    print("Nessun file valido per Asia!")

# 6. Report mancanti
if missing_files:
    print("\n=== FILE MANCANTI ===")
    print(f"Totale: {len(missing_files)}")
    for m in missing_files:
        print(m)
else:
    print("Tutti i file presenti o processati correttamente!")

In [0]:
# =============================================================================
# SINGOLO CODICE: MAPPATURA + ESTRAZIONE DATI SOLO EUROPA (2000-2024)
# =============================================================================

from pyspark.sql import functions as F
import time
import random

# 1. Mappatura SOLO EUROPA (5 paesi × 3 città = 15 stazioni)
europa_stations = {
    "Europa": {
        "Italia": [
            {"code": "160200", "city": "Bolzano (alpino/freddo)"},
            {"code": "162390", "city": "Roma Ciampino (mediterraneo)"},
            {"code": "164050", "city": "Palermo Punta Raisi (subtropicale/caldo)"}
        ],
        "Francia": [
            {"code": "071300", "city": "Parigi Charles de Gaulle"},
            {"code": "076500", "city": "Marsiglia"},
            {"code": "071490", "city": "Brest (oceanico)"}
        ],
        "Germania": [
            {"code": "103840", "city": "Berlino Tempelhof"},
            {"code": "108700", "city": "Monaco"},
            {"code": "101470", "city": "Amburgo"}
        ],
        "Spagna": [
            {"code": "080010", "city": "Madrid Barajas"},
            {"code": "083140", "city": "Barcellona"},
            {"code": "081910", "city": "Malaga (caldo)"}
        ],
        "Regno Unito": [
            {"code": "037720", "city": "Londra Heathrow"},
            {"code": "038080", "city": "Birmingham"},
            {"code": "030910", "city": "Aberdeen (freddo/nord)"}
        ]
    }
}

# Flatten + mappa code → info città
usaf_to_city = {}
for paese, lista in europa_stations["Europa"].items():
    for item in lista:
        code = item["code"]
        usaf_to_city[code] = {
            "paese": paese,
            "city": item["city"]
        }

usaf_list = list(usaf_to_city.keys())
print(f"Stazioni Europa selezionate: {len(usaf_list)}")
print("Codici:", usaf_list)

# 2. Genera percorsi S3 (2000-2024)
years = list(range(2000, 2025))
paths_with_info = []
for year in years:
    for code in usaf_list:
        path = f"s3a://noaa-gsod-pds/{year}/{code}99999.csv"
        paths_with_info.append((path, code, year))

print(f"Totale percorsi generati: {len(paths_with_info)}")

# 3. Lettura sicura con try/except + anti-throttling
dfs = []
missing_files = []

for path, code, year in paths_with_info:
    time.sleep(random.uniform(0.4, 1.5))  # evita throttling S3
    try:
        df_temp = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(path)
        
        if df_temp.count() > 0:
            df_temp = df_temp.withColumn("USAF", F.lit(code)) \
                             .withColumn("year", F.lit(year))
            dfs.append(df_temp)
            print(f"✓ Letto: {path}")
        else:
            city = usaf_to_city.get(code, {}).get("city", "Sconosciuta")
            missing_files.append(f"{year} - {code} ({city}) → file vuoto")
    except Exception as e:
        city = usaf_to_city.get(code, {}).get("city", "Sconosciuta")
        missing_files.append(f"{year} - {code} ({city}) → {str(e)[:80]}...")

# 4. Unione dati validi
if dfs:
    df_europa = dfs[0]
    for d in dfs[1:]:
        df_europa = df_europa.unionByName(d, allowMissingColumns=True)
    
    # Aggiungi metadati
    df_europa = df_europa.withColumn("continent", F.lit("Europa")) \
                         .withColumn("season",
                                     F.when(F.month("DATE").isin([12,1,2]), "Inverno")
                                      .when(F.month("DATE").isin([3,4,5]), "Primavera")
                                      .when(F.month("DATE").isin([6,7,8]), "Estate")
                                      .otherwise("Autunno"))
    
    # Controllo temperature (converti se in °F)
    temp_avg = df_europa.agg(F.avg("TEMP").alias("avg")).collect()[0]["avg"]
    if temp_avg is not None and temp_avg > 40:
        print("→ Temperature in °F → conversione in °C")
        df_europa = df_europa.withColumn("TEMP_c",   (F.col("TEMP") - 32) * 5/9) \
                             .withColumn("MAX_c",    (F.col("MAX") - 32) * 5/9) \
                             .withColumn("MIN_c",    (F.col("MIN") - 32) * 5/9)
    else:
        df_europa = df_europa.withColumnRenamed("TEMP", "TEMP_c") \
                             .withColumnRenamed("MAX", "MAX_c") \
                             .withColumnRenamed("MIN", "MIN_c")

    # Filtra righe valide
    df_europa_clean = df_europa.filter(F.col("TEMP_c").isNotNull())
    
    print(f"\nEuropa completata — righe totali: {df_europa_clean.count()}")
  
    
    # 5. Salva in Delta (adatta al tuo Volume!)
    delta_path = "/Volumes/workspace/weather/continents/eu/"  # ← CAMBIA CON IL TUO VOLU
    df_europa_clean.write.format("delta").mode("overwrite").save(delta_path)
    print(f"Salvato in Delta: {delta_path}")
    
else:
    print("Nessun file valido per Europa!")   

# 6. Report mancanti
if missing_files:
    print("\n=== FILE MANCANTI ===")
    print(f"Totale: {len(missing_files)}")
    for m in missing_files:
        print(m)
else:
    print("Tutti i file presenti o processati correttamente!")

In [0]:
# =============================================================================
# SINGOLO CODICE: MAPPATURA + ESTRAZIONE DATI SOLO NORD AMERICA (2000-2024)
# =============================================================================

from pyspark.sql import functions as F
import time
import random

# 1. Mappatura SOLO NORD AMERICA (5 paesi × 3 città = 15 stazioni)
nordamerica_stations = {
    "Nord America": {
        "Stati Uniti": [
            {"code": "725030", "city": "New York JFK (temperato umido est)"},
            {"code": "722950", "city": "Los Angeles (mediterraneo costiero)"},
            {"code": "722740", "city": "Miami (tropicale umido sud)"}
        ],
        "Canada": [
            {"code": "716240", "city": "Toronto (continentale freddo)"},
            {"code": "718920", "city": "Vancouver (oceanico mite)"},
            {"code": "719360", "city": "Montreal (freddo continentale)"}
        ],
        "Messico": [
            {"code": "766790", "city": "Città del Messico (altopiano subtropicale)"},
            {"code": "764580", "city": "Monterrey (caldo arido nord)"},
            {"code": "766440", "city": "Cancun (tropicale caraibico)"}
        ],
        "Cuba": [
            {"code": "783970", "city": "L'Avana (tropicale umido)"},
            {"code": "783250", "city": "Santiago de Cuba (tropicale sud)"}
        ],
        "Guatemala": [
            {"code": "786660", "city": "Città del Guatemala (altopiano tropicale)"}
        ]
    }
}

# Flatten + mappa code → info città
usaf_to_city = {}
for paese, lista in nordamerica_stations["Nord America"].items():
    for item in lista:
        code = item["code"]
        usaf_to_city[code] = {
            "paese": paese,
            "city": item["city"]
        }

usaf_list = list(usaf_to_city.keys())
print(f"Stazioni Nord America selezionate: {len(usaf_list)}")
print("Codici:", usaf_list)

# 2. Genera percorsi S3 (2000-2024)
years = list(range(2000, 2025))
paths_with_info = []
for year in years:
    for code in usaf_list:
        path = f"s3a://noaa-gsod-pds/{year}/{code}99999.csv"
        paths_with_info.append((path, code, year))

print(f"Totale percorsi generati: {len(paths_with_info)}")

# 3. Lettura sicura con try/except + anti-throttling
dfs = []
missing_files = []

for path, code, year in paths_with_info:
    time.sleep(random.uniform(0.4, 1.5))  # evita throttling S3
    try:
        df_temp = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(path)
        
        if df_temp.count() > 0:
            df_temp = df_temp.withColumn("USAF", F.lit(code)) \
                             .withColumn("year", F.lit(year))
            dfs.append(df_temp)
            print(f"✓ Letto: {path}")
        else:
            city = usaf_to_city.get(code, {}).get("city", "Sconosciuta")
            missing_files.append(f"{year} - {code} ({city}) → file vuoto")
    except Exception as e:
        city = usaf_to_city.get(code, {}).get("city", "Sconosciuta")
        missing_files.append(f"{year} - {code} ({city}) → {str(e)[:80]}...")

# 4. Unione dati validi
if dfs:
    df_nordamerica = dfs[0]
    for d in dfs[1:]:
        df_nordamerica = df_nordamerica.unionByName(d, allowMissingColumns=True)
    
    # Aggiungi metadati
    df_nordamerica = df_nordamerica.withColumn("continent", F.lit("Nord America")) \
                                  .withColumn("season",
                                              F.when(F.month("DATE").isin([12,1,2]), "Inverno")
                                               .when(F.month("DATE").isin([3,4,5]), "Primavera")
                                               .when(F.month("DATE").isin([6,7,8]), "Estate")
                                               .otherwise("Autunno"))
    
    # Controllo temperature (converti se in °F)
    temp_avg = df_nordamerica.agg(F.avg("TEMP").alias("avg")).collect()[0]["avg"]
    if temp_avg is not None and temp_avg > 40:
        print("→ Temperature in °F → conversione in °C")
        df_nordamerica = df_nordamerica.withColumn("TEMP_c",   (F.col("TEMP") - 32) * 5/9) \
                                       .withColumn("MAX_c",    (F.col("MAX") - 32) * 5/9) \
                                       .withColumn("MIN_c",    (F.col("MIN") - 32) * 5/9)
    else:
        df_nordamerica = df_nordamerica.withColumnRenamed("TEMP", "TEMP_c") \
                                       .withColumnRenamed("MAX", "MAX_c") \
                                       .withColumnRenamed("MIN", "MIN_c")

    # Filtra righe valide
    df_nordamerica_clean = df_nordamerica.filter(F.col("TEMP_c").isNotNull())
    
    print(f"\nNord America completata — righe totali: {df_nordamerica_clean.count()}")

    
    # 5. Salva in Delta (adatta al tuo Volume!)
    delta_path = "/Volumes/workspace/weather/continents/na/"  # ← CAMBIA CON IL TUO VO
    df_nordamerica_clean.write.format("delta").mode("overwrite").save(delta_path)
    print(f"Salvato in Delta: {delta_path}")
    
else:
    print("Nessun file valido per Nord America!")

# 6. Report mancanti
if missing_files:
    print("\n=== FILE MANCANTI ===")
    print(f"Totale: {len(missing_files)}")
    for m in missing_files:
        print(m)
else:
    print("Tutti i file presenti o processati correttamente!")

In [0]:
# =============================================================================
# SINGOLO CODICE: MAPPATURA + ESTRAZIONE DATI SOLO AMERICA DEL SUD (2000-2024)
# =============================================================================

from pyspark.sql import functions as F
import time
import random

# 1. Mappatura SOLO AMERICA DEL SUD (5 paesi × 3 città = 15 stazioni)
sudamerica_stations = {
    "Sud America": {
        "Brasile": [
            {"code": "837800", "city": "Rio de Janeiro (tropicale atlantico)"},
            {"code": "839070", "city": "San Paolo (subtropicale altopiano)"},
            {"code": "823320", "city": "Manaus (amazzonico equatoriale)"}
        ],
        "Argentina": [
            {"code": "876330", "city": "Buenos Aires (temperato pampano)"},
            {"code": "871780", "city": "Cordoba (continentale)"},
            {"code": "877650", "city": "Ushuaia (freddo subpolare sud)"}
        ],
        "Cile": [
            {"code": "855860", "city": "Santiago (mediterraneo andino)"},
            {"code": "854700", "city": "Punta Arenas (freddo patagonico)"}
        ],
        "Colombia": [
            {"code": "802590", "city": "Bogotà (altopiano tropicale)"},
            {"code": "802220", "city": "Cartagena (caraibico caldo)"}
        ],
        "Perù": [
            {"code": "846280", "city": "Lima (deserto costiero)"},
            {"code": "846860", "city": "Cuzco (andino alto)"}
        ]
    }
}

# Flatten + mappa code → info città
usaf_to_city = {}
for paese, lista in sudamerica_stations["Sud America"].items():
    for item in lista:
        code = item["code"]
        usaf_to_city[code] = {
            "paese": paese,
            "city": item["city"]
        }

usaf_list = list(usaf_to_city.keys())
print(f"Stazioni Sud America selezionate: {len(usaf_list)}")
print("Codici:", usaf_list)

# 2. Genera percorsi S3 (2000-2024)
years = list(range(2000, 2025))
paths_with_info = []
for year in years:
    for code in usaf_list:
        path = f"s3a://noaa-gsod-pds/{year}/{code}99999.csv"
        paths_with_info.append((path, code, year))

print(f"Totale percorsi generati: {len(paths_with_info)}")

# 3. Lettura sicura con try/except + anti-throttling
dfs = []
missing_files = []

for path, code, year in paths_with_info:
    time.sleep(random.uniform(0.4, 1.5))  # evita throttling S3
    try:
        df_temp = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(path)
        
        if df_temp.count() > 0:
            df_temp = df_temp.withColumn("USAF", F.lit(code)) \
                             .withColumn("year", F.lit(year))
            dfs.append(df_temp)
            print(f"✓ Letto: {path}")
        else:
            city = usaf_to_city.get(code, {}).get("city", "Sconosciuta")
            missing_files.append(f"{year} - {code} ({city}) → file vuoto")
    except Exception as e:
        city = usaf_to_city.get(code, {}).get("city", "Sconosciuta")
        missing_files.append(f"{year} - {code} ({city}) → {str(e)[:80]}...")

# 4. Unione dati validi
if dfs:
    df_sudamerica = dfs[0]
    for d in dfs[1:]:
        df_sudamerica = df_sudamerica.unionByName(d, allowMissingColumns=True)
    
    # Aggiungi metadati
    df_sudamerica = df_sudamerica.withColumn("continent", F.lit("Sud America")) \
                                 .withColumn("season",
                                             F.when(F.month("DATE").isin([12,1,2]), "Inverno")
                                              .when(F.month("DATE").isin([3,4,5]), "Primavera")
                                              .when(F.month("DATE").isin([6,7,8]), "Estate")
                                              .otherwise("Autunno"))
    
    # Controllo temperature (converti se in °F)
    temp_avg = df_sudamerica.agg(F.avg("TEMP").alias("avg")).collect()[0]["avg"]
    if temp_avg is not None and temp_avg > 40:
        print("→ Temperature in °F → conversione in °C")
        df_sudamerica = df_sudamerica.withColumn("TEMP_c",   (F.col("TEMP") - 32) * 5/9) \
                                     .withColumn("MAX_c",    (F.col("MAX") - 32) * 5/9) \
                                     .withColumn("MIN_c",    (F.col("MIN") - 32) * 5/9)
    else:
        df_sudamerica = df_sudamerica.withColumnRenamed("TEMP", "TEMP_c") \
                                     .withColumnRenamed("MAX", "MAX_c") \
                                     .withColumnRenamed("MIN", "MIN_c")

    # Filtra righe valide
    df_sudamerica_clean = df_sudamerica.filter(F.col("TEMP_c").isNotNull())
    
    print(f"\nSud America completata — righe totali: {df_sudamerica_clean.count()}")
    
    # 5. Salva in Delta (adatta al tuo Volume!)
    delta_path = "/Volumes/workspace/weather/continents/sa/"  # ← CAMBIA CON IL TUO VOLUM
    df_sudamerica_clean.write.format("delta").mode("overwrite").save(delta_path)
    print(f"Salvato in Delta: {delta_path}")
    
else:
    print("Nessun file valido per Sud America!")

# 6. Report mancanti
if missing_files:
    print("\n=== FILE MANCANTI ===")
    print(f"Totale: {len(missing_files)}")
    for m in missing_files:
        print(m)
else:
    print("Tutti i file presenti o processati correttamente!")

In [0]:
# =============================================================================
# SINGOLO CODICE: MAPPATURA + ESTRAZIONE DATI SOLO OCEANIA (2000-2024)
# =============================================================================

from pyspark.sql import functions as F
import time
import random

# 1. Mappatura SOLO OCEANIA (Australia + Nuova Zelanda + 1 extra per diversità)
oceania_stations = {
    "Oceania": {
        "Australia": [
            {"code": "946720", "city": "Sydney (costiero temperato umido)"},
            {"code": "948660", "city": "Melbourne (temperato oceanico)"},
            {"code": "946380", "city": "Perth (mediterraneo arido ovest)"},
            {"code": "942990", "city": "Darwin (tropicale monsonica nord)"},
            {"code": "946370", "city": "Adelaide (mediterraneo secco)"}
        ],
        "Nuova Zelanda": [
            {"code": "936140", "city": "Auckland (oceanico temperato)"},
            {"code": "934170", "city": "Christchurch (temperato fresco)"},
            {"code": "936780", "city": "Wellington (ventoso oceanico)"}
        ],
        "Papua Nuova Guinea": [  # extra per diversità tropicale
            {"code": "940840", "city": "Port Moresby (tropicale umido)"}
        ]
    }
}

# Flatten + mappa code → info città
usaf_to_city = {}
for paese, lista in oceania_stations["Oceania"].items():
    for item in lista:
        code = item["code"]
        usaf_to_city[code] = {
            "paese": paese,
            "city": item["city"]
        }

usaf_list = list(usaf_to_city.keys())
print(f"Stazioni Oceania selezionate: {len(usaf_list)}")
print("Codici:", usaf_list)

# 2. Genera percorsi S3 (2000-2024)
years = list(range(2000, 2025))
paths_with_info = []
for year in years:
    for code in usaf_list:
        path = f"s3a://noaa-gsod-pds/{year}/{code}99999.csv"
        paths_with_info.append((path, code, year))

print(f"Totale percorsi generati: {len(paths_with_info)}")

# 3. Lettura sicura con try/except + anti-throttling
dfs = []
missing_files = []

for path, code, year in paths_with_info:
    time.sleep(random.uniform(0.4, 1.5))  # evita throttling S3
    try:
        df_temp = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(path)
        
        if df_temp.count() > 0:
            df_temp = df_temp.withColumn("USAF", F.lit(code)) \
                             .withColumn("year", F.lit(year))
            dfs.append(df_temp)
            print(f"✓ Letto: {path}")
        else:
            city = usaf_to_city.get(code, {}).get("city", "Sconosciuta")
            missing_files.append(f"{year} - {code} ({city}) → file vuoto")
    except Exception as e:
        city = usaf_to_city.get(code, {}).get("city", "Sconosciuta")
        missing_files.append(f"{year} - {code} ({city}) → {str(e)[:80]}...")

# 4. Unione dati validi
if dfs:
    df_oceania = dfs[0]
    for d in dfs[1:]:
        df_oceania = df_oceania.unionByName(d, allowMissingColumns=True)
    
    # Aggiungi metadati
    df_oceania = df_oceania.withColumn("continent", F.lit("Oceania")) \
                           .withColumn("season",
                                       F.when(F.month("DATE").isin([12,1,2]), "Inverno")
                                        .when(F.month("DATE").isin([3,4,5]), "Primavera")
                                        .when(F.month("DATE").isin([6,7,8]), "Estate")
                                        .otherwise("Autunno"))
    
    # Controllo temperature (converti se in °F)
    temp_avg = df_oceania.agg(F.avg("TEMP").alias("avg")).collect()[0]["avg"]
    if temp_avg is not None and temp_avg > 40:
        print("→ Temperature in °F → conversione in °C")
        df_oceania = df_oceania.withColumn("TEMP_c",   (F.col("TEMP") - 32) * 5/9) \
                               .withColumn("MAX_c",    (F.col("MAX") - 32) * 5/9) \
                               .withColumn("MIN_c",    (F.col("MIN") - 32) * 5/9)
    else:
        df_oceania = df_oceania.withColumnRenamed("TEMP", "TEMP_c") \
                               .withColumnRenamed("MAX", "MAX_c") \
                               .withColumnRenamed("MIN", "MIN_c")

    # Filtra righe valide
    df_oceania_clean = df_oceania.filter(F.col("TEMP_c").isNotNull())
    
    print(f"\nOceania completata — righe totali: {df_oceania_clean.count()}")
    
    # 5. Salva in Delta (adatta al tuo Volume!)
    delta_path = "/Volumes/workspace/weather/continents/oc"  # ← CAMBIA CON IL TUO VOLUM
    df_oceania_clean.write.format("delta").mode("overwrite").save(delta_path)
    print(f"Salvato in Delta: {delta_path}")
    
else:
    print("Nessun file valido per Oceania!")

# 6. Report mancanti
if missing_files:
    print("\n=== FILE MANCANTI ===")
    print(f"Totale: {len(missing_files)}")
    for m in missing_files:
        print(m)
else:
    print("Tutti i file presenti o processati correttamente!")